### Load golden answers

In [ ]:
import json

with open('data/ptpt-multiple-choice-qa-pairs.json', 'r', encoding='utf-8') as f:
    golden_qa_pairs = json.load(f)

### Load model answers

In [ ]:
responses = []
with open('results/Qwen3-4B-ptpt-responses.json', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():  # skip empty lines
            responses.append(json.loads(line))

### Parse model final answers

In [ ]:
import re

nr_responses = 0

for response in responses:
    try:
        text = response["raw_response"]["choice.message.content"]
    except KeyError:
        # skip responses without the content field
        response['final_answer'] = None
        continue

    m = re.search(r'(?<=\\boxed\{)([A-Za-z])(?=\}(?!.*\\boxed))', text)
    if m:
        response["final_answer"] = m.group(0)   # store the letter string, not the match object
        nr_responses += 1
    else:
        response["final_answer"] = None
        # optional: print("No match for id", response.get('id'))
        
print(f"Number of responses processed: {nr_responses}")


### Calculate accuracies

In [ ]:
import re
from collections import defaultdict

# build lookup tables
responses_by_id = {r.get('id'): r for r in responses}
golden_by_id = {g['id']: g for g in golden_qa_pairs}

# initialize counters per level (levels 1..4)
levels = [1, 2, 3, 4]
stats = {
    lvl: {
        'total_gold': 0,            # total golden pairs for this level
        'answered': 0,             # number of responses that provided a final_answer
        'correct_answered': 0,     # correct among answered
        'correct_including_missing': 0  # correct when missing counted as incorrect
    }
    for lvl in levels
}

# iterate gold entries (ensures we only evaluate ids that have gold answers)
for gid, golden in golden_by_id.items():
    # parse level (be defensive: might be string)
    try:
        lvl = int(golden.get('level', 0))
    except (TypeError, ValueError):
        continue
    if lvl not in levels:
        continue

    stats[lvl]['total_gold'] += 1

    resp = responses_by_id.get(gid)
    final_ans = None

    if resp is not None:
        # final_answer might be a re.Match or a plain string or None
        fa = resp.get('final_answer', None)
        if fa is not None:
            if hasattr(fa, 'group'):   # re.Match-like
                try:
                    final_ans = fa.group(0)
                except Exception:
                    final_ans = None
            else:
                final_ans = fa  # assume already a string

    correct_option = str(golden.get('correct_option', '')).strip().lower()

    if final_ans is not None:
        stats[lvl]['answered'] += 1
        # safe-normalize final answer
        final_norm = str(final_ans).strip().lower()
        if final_norm == correct_option:
            stats[lvl]['correct_answered'] += 1
            stats[lvl]['correct_including_missing'] += 1
    else:
        # missing answer counts as incorrect for 'including_missing' metric,
        # so do nothing to correct_including_missing
        pass

# compute global aggregates
global_totals = {
    'total_gold': sum(stats[l]['total_gold'] for l in levels),
    'answered': sum(stats[l]['answered'] for l in levels),
    'correct_answered': sum(stats[l]['correct_answered'] for l in levels),
    'correct_including_missing': sum(stats[l]['correct_including_missing'] for l in levels)
}

# print nicely
def percent(num, denom):
    return f"{(num/denom*100):.2f}%" if denom else "N/A"

print("Per-level results (levels 1..4):\n")
for l in levels:
    s = stats[l]
    print(f"Level {l}:")
    print(f"  Golden items (denominator for 'including missing') : {s['total_gold']}")
    print(f"  Answered (have final_answer)                          : {s['answered']}")
    print(f"  Correct among answered                               : {s['correct_answered']}")
    print(f"  Accuracy (answered only)                             : {percent(s['correct_answered'], s['answered'])}")
    print(f"  Accuracy (missing counted as incorrect)              : {percent(s['correct_including_missing'], s['total_gold'])}")
    print()

print("GLOBAL (all levels combined):")
print(f"  Golden items total: {global_totals['total_gold']}")
print(f"  Answered total     : {global_totals['answered']}")
print(f"  Correct (answered) : {global_totals['correct_answered']}")
print(f"  Accuracy (answered only)         : {percent(global_totals['correct_answered'], global_totals['answered'])}")
print(f"  Accuracy (missing counted wrong) : {percent(global_totals['correct_including_missing'], global_totals['total_gold'])}")
